# Information diagnostics for fitted edge BPE

This notebook is deliberately **external** to `tree_coarsening`.  It uses a fitted
`EdgeBPECoarsener` and the companion file `experiments/information_diagnostics.py`.
It explores two different questions:

1. **Path-level compression:** after 0, 1, 2, … fitted rules, how many vertices remain,
and how does a transparent held-out description-length surrogate change?
2. **Occurrence-level surprise:** after transforming a new tree, which compressed
vertices/tokens are unexpected under the transformed training distribution?

The distinction matters.  A learned token type can have a common construction recipe,
while one particular occurrence of that token can still be surprising in its current
parent context.  Conversely, a rare learned recipe can occur in an unsurprising place.
The notebook keeps those quantities separate rather than adding them automatically.


## Statistical quantities

At BPE stage $t$, let $X_v^{(t)}$ be the current token at vertex $v$ of a rooted
directed tree $T=(V,E)$, and let $r$ be the root.  A first-order tree-indexed
Markov model factors as

$$
q_t(x_V)
= q_{R,t}(x_r)
  \prod_{(u,v)\in E}q_t(x_v\mid x_u).
$$

For a corpus, the proper edge-sampled conditional maximum-likelihood estimate is

$$
\widehat p(b\mid a)
= \frac{N_{ab}}{N_{a\to\bullet}},
\qquad
N_{a\to\bullet}=\sum_c N_{ac}.
$$

This denominator is **not** generally the number of vertices labelled $a$.
On a branching tree, $N_{ab}/N_a^{\mathrm{vertices}}$ can even exceed one because
one parent can contribute many outgoing edges.

The experiment uses hierarchical Dirichlet smoothing,

$$
q(b\mid a)
= \frac{N_{ab}+\alpha p_0(b)}{N_{a\to\bullet}+\alpha},
$$

where $p_0$ is a smoothed global child-token distribution.  `weighting="edge"`
lets every edge contribute one count.  `weighting="parent"` instead uses

$$
\widetilde N_{ab}
= \sum_{u:X_u=a,\ d_u^+>0}
  \frac{\#\{v:u\to v,\ X_v=b\}}{d_u^+},
$$

so every non-leaf parent contributes total mass one; this is often useful when a
few huge hubs would otherwise dominate the estimate.

For a parent with timestamp-ordered children $b_1,\ldots,b_d$, the optional
sibling-predictive score is

$$
q(b_j\mid a,b_{<j})
=
\frac{N_{ab_j}+\alpha p_0(b_j)+C_{<j}(b_j)}
     {N_{a\to\bullet}+\alpha+j-1}.
$$

The product over the child bag is exchangeable: its total is independent of the
chosen order, although individual child contributions are reported in deterministic
timestamp order.

For stage comparison, the notebook uses the explicit two-part surrogate

$$
L_t
= L_{\mathrm{labels},t}
+ L_{\mathrm{shape},t}
+ L_{\mathrm{rules},t},
\qquad
G_t=L_{t-1}-L_t.
$$

Here $L_{\mathrm{labels},t}$ is either the transition or sibling-predictive code,
$L_{\mathrm{shape},t}=\sum_i\log_2 C_{|V_i|-1}$ uses the Catalan count of ordered
rooted-tree shapes, and

$$
L_{\mathrm{rules},t}
=\sum_{r=0}^{t-1}2\log_2(K_0+r)
$$

is a simple edge-rule-table cost.  These are interpretable surrogates, not a claim
of an optimal universal code.  Validation-corpus changes in $L_t$ are usually more
useful than in-sample changes.

For one held-out non-root occurrence,

$$
I(v)=-\log_2 q(X_v\mid X_{\operatorname{pa}(v)})
$$

is contextual surprisal.  The association contrast

$$
A(v)=\log_2\frac{q(X_v\mid X_{\operatorname{pa}(v)})}{p_0(X_v)}
$$

answers a different question: whether the child is more likely in that parent
context than globally.


### References

- C. E. Shannon, *A Mathematical Theory of Communication* (1948):
  [DOI 10.1002/j.1538-7305.1948.tb01338.x](https://doi.org/10.1002/j.1538-7305.1948.tb01338.x).
- I. Benjamini and Y. Peres, *Markov Chains Indexed by Trees* (1994):
  [DOI 10.1214/aop/1176988857](https://doi.org/10.1214/aop/1176988857).
- R. E. Krichevsky and V. K. Trofimov, *The Performance of Universal Encoding*
  (1981), for Dirichlet/KT predictive coding:
  [DOI 10.1109/TIT.1981.1056331](https://doi.org/10.1109/TIT.1981.1056331).
- J. Rissanen, *Modeling by Shortest Data Description* (1978):
  [DOI 10.1016/0005-1098(78)90005-5](https://doi.org/10.1016/0005-1098(78)90005-5).
- The fitted API used here is from
  [`AMS-Hippo/tree_condenser_stamp`](https://github.com/AMS-Hippo/tree_condenser_stamp).


In [1]:
from __future__ import annotations

from pathlib import Path
import sys

import matplotlib.pyplot as plt
import networkx as nx
import pandas as pd

# Locate the external experiment file whether the kernel starts in the repository
# root, in notebooks/, or in experiments/.
search_roots = [Path.cwd(), Path.cwd().parent]
experiment_dir = next(
    (
        candidate
        for root in search_roots
        for candidate in (root / "experiments", root)
        if (candidate / "information_diagnostics.py").exists()
    ),
    None,
)
if experiment_dir is None:
    raise FileNotFoundError(
        "Place information_diagnostics.py in the repository's experiments/ "
        "directory, then run this notebook from the repository root or notebooks/."
    )
sys.path.insert(0, str(experiment_dir))

import tree_coarsening
from tree_coarsening import (
    EdgeBPECoarsener,
    make_edge_bpe_dataset,
    make_repeated_edge_tree,
    make_starburst_dataset,
)
from information_diagnostics import (
    TreeMarkovModel,
    analyze_edge_bpe_path,
    fit_transformed_markov_model,
)

print("tree_coarsening imported from:", tree_coarsening.__file__)
print("experiment file:", experiment_dir / "information_diagnostics.py")


ImportError: cannot import name 'make_edge_bpe_dataset' from 'tree_coarsening' (/home/asmi28/anaconda3/envs/steering/lib/python3.13/site-packages/tree_coarsening/__init__.py)

In [ ]:
def format_token(token: object) -> str:
    """Return a compact human-readable representation of a token."""
    if (
        isinstance(token, tuple)
        and len(token) == 2
        and token[0] == "base"
    ):
        return f"base:{token[1]}"

    if isinstance(token, tuple):
        return ":".join(str(part) for part in token)

    return str(token)

## 1. Generate training, validation, and test trees

The training corpus mixes repeated paths with high-degree starbursts.  The path
motifs make hierarchical edge-BPE tokens easy to see; the starbursts make the
difference between edge-weighted and parent-balanced estimates meaningful.


In [ ]:
train = [
    *make_edge_bpe_dataset(
        n_graphs=24,
        n_repeats=32,
        motif_labels=("A", "B", "C", "D"),
        anchor_labels=("X", "Y", "Z", "W"),
        seed=10,
    ),
    *make_starburst_dataset(
        n_graphs=12,
        max_nodes=45,
        n_bursts=3,
        burst_size_range=(8, 18),
        parent_label="P",
        child_label="S",
        tail_label="T",
        tail_probability=0.35,
        labels=("A", "B", "C", "D"),
        seed=20,
    ),
]

validation = [
    *make_edge_bpe_dataset(
        n_graphs=8,
        n_repeats=28,
        motif_labels=("A", "B", "C", "D"),
        anchor_labels=("X", "Y", "Z", "W"),
        seed=30,
    ),
    *make_starburst_dataset(
        n_graphs=4,
        max_nodes=45,
        n_bursts=3,
        burst_size_range=(8, 18),
        parent_label="P",
        child_label="S",
        tail_label="T",
        tail_probability=0.35,
        labels=("A", "B", "C", "D"),
        seed=40,
    ),
]

def corpus_size(graphs: list[nx.DiGraph]) -> tuple[int, int]:
    return (
        sum(graph.number_of_nodes() for graph in graphs),
        sum(graph.number_of_edges() for graph in graphs),
    )

print("training (vertices, edges):", corpus_size(train))
print("validation (vertices, edges):", corpus_size(validation))


## 2. Fit ordinary edge BPE

The information code does not change fitting.  It consumes the fitted public
encoder artifact and `history_` after the normal `fit` call.


In [ ]:
coarsener = EdgeBPECoarsener(
    num_merges=14,
    min_pair_count=4,
    pair_score="count",
    backend="python",
).fit(train)

print("rules learned:", len(coarsener.encoder_.edge_rules))
pd.DataFrame(coarsener.history_).head(10)


## 3. Replay the fitted rule path

`analyze_edge_bpe_path` applies the already-learned encoder rules one at a time.
At each stage it refits the small probability model on the current transformed
training representation and scores both training and validation representations.
This is intentionally simple rather than optimized.

The returned gain is **realized code-length reduction** under the chosen surrogate,
not the original BPE pair score and not a theorem about optimal compression.


In [ ]:
path = analyze_edge_bpe_path(
    coarsener,
    train,
    validation_graphs=validation,
    alpha=8.0,
    base_pseudocount=0.5,
    transition_weighting="edge",
    topology="catalan",
    validate_replay=False,
)

stage_df = pd.DataFrame(path.stage_records())
rule_df = pd.DataFrame(path.rule_records())

for split in ("train", "validation"):
    sites = stage_df[f"{split}_represented_sites"]
    stage_df[f"{split}_transition_total_bits_per_site"] = (
        stage_df[f"{split}_transition_total_bits"] / sites
    )
    stage_df[f"{split}_sibling_total_bits_per_site"] = (
        stage_df[f"{split}_sibling_total_bits"] / sites
    )

stage_df[[
    "stage",
    "train_n_nodes",
    "validation_n_nodes",
    "train_compression_rate",
    "validation_compression_rate",
    "validation_transition_total_bits_per_site",
    "validation_sibling_total_bits_per_site",
    "validation_transition_gain_bits",
    "validation_sibling_gain_bits",
]].round(4)


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(stage_df["stage"], stage_df["train_compression_rate"], marker="o", label="train")
ax.plot(stage_df["stage"], stage_df["validation_compression_rate"], marker="o", label="validation")
ax.set(xlabel="number of fitted rules applied", ylabel="vertex compression rate", title="Structural compression path")
ax.legend()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(
    stage_df["stage"],
    stage_df["validation_transition_total_bits_per_site"],
    marker="o",
    label="first-order transition code",
)
ax.plot(
    stage_df["stage"],
    stage_df["validation_sibling_total_bits_per_site"],
    marker="o",
    label="sibling-predictive code",
)
ax.set(
    xlabel="number of fitted rules applied",
    ylabel="validation bits per represented original vertex",
    title="Held-out description-length surrogates",
)
ax.legend()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(
    stage_df["stage"],
    stage_df["validation_transition_gain_bits"],
    marker="o",
    label="transition-code gain",
)
ax.plot(
    stage_df["stage"],
    stage_df["validation_sibling_gain_bits"],
    marker="o",
    label="sibling-code gain",
)
ax.axhline(0.0, linewidth=1)
ax.set(
    xlabel="stage",
    ylabel="previous bits minus current bits",
    title="Incremental held-out gain (positive is better)",
)
ax.legend()
plt.show()


The incremental curve can be non-monotone.  For model selection, inspect the
**cumulative held-out objective** and choose a robust prefix (for example, a smoothed
minimum with patience/backtracking), rather than stopping at the first negative step.
The stage table also exposes gain per removed node when that denominator is positive.


In [ ]:
rule_view = rule_df[[
    "rank",
    "token",
    "parent_label",
    "child_label",
    "raw_count",
    "fit_actual_events",
    "replay_train_events",
    "replay_validation_events",
    "smoothed_transition_probability",
    "local_surprisal_bits",
    "local_association_bits",
    "construction_surprisal_bits",
    "construction_surprisal_per_site_bits",
]].copy()
rule_view["token"] = rule_view["token"].map(format_token)
rule_view["parent_label"] = rule_view["parent_label"].map(format_token)
rule_view["child_label"] = rule_view["child_label"].map(format_token)
rule_view.round(4)


`local_surprisal_bits` measures the selected edge recipe at the stage when the
rule was learned.  `construction_surprisal_bits` recursively includes recipes of
nested BPE children.  `local_association_bits` is not surprisal: a positive value
means the pair is enriched relative to the child's global frequency.


## 4. Why $N_{ab}/N_a^{\mathrm{vertices}}$ is not a transition probability

The mixed training corpus contains large stars.  The following table compares the
naive vertex denominator with the proper outgoing-edge denominator before any BPE
rules are applied.


In [ ]:
raw_model = TreeMarkovModel(alpha=8.0, weighting="edge").fit(train)
ratio_rows = []
for parent_label, child_counts in raw_model.transition_counts_.items():
    for child_label, n_ab in child_counts.items():
        n_parent_vertices = raw_model.vertex_counts_[parent_label]
        n_outgoing = raw_model.context_totals_[parent_label]
        ratio_rows.append(
            {
                "parent": parent_label,
                "child": child_label,
                "N_ab": n_ab,
                "N_parent_vertices": n_parent_vertices,
                "N_outgoing_from_parent_label": n_outgoing,
                "naive_Nab_over_parent_vertices": n_ab / n_parent_vertices,
                "proper_edge_MLE": n_ab / n_outgoing,
                "smoothed_q": raw_model.transition_probability(parent_label, child_label),
            }
        )

ratio_df = pd.DataFrame(ratio_rows).sort_values(
    "naive_Nab_over_parent_vertices", ascending=False
)
ratio_df.head(12).round(4)


A naive ratio above one is not a numerical bug; it demonstrates that the denominator
was counting the wrong sampling unit.  The edge-MLE column is a valid conditional
probability.  The smoothed column avoids zero probability on held-out edges.


## 5. Score a transformed held-out tree

We inject one structurally unusual edge using only labels already present in training:
one branch that normally contains `A → B` is changed to `A → X`.  This avoids
confounding contextual surprise with an entirely unseen alphabet symbol.


In [ ]:
test = make_repeated_edge_tree(
    n_repeats=18,
    motif_labels=("A", "B", "C", "D"),
    anchor_labels=("X", "Y", "Z", "W"),
    seed=99,
    uid_prefix="heldout",
)

injected_node = next(
    node
    for node in nx.topological_sort(test)
    if test.in_degree(node) == 1
    and test.nodes[next(test.predecessors(node))]["label"] == "A"
    and test.nodes[node]["label"] == "B"
)
injected_uid = test.nodes[injected_node]["uid"]
test.nodes[injected_node]["label"] = "X"

H_test = coarsener.transform(test)
injected_token_node = next(
    node
    for node, data in H_test.nodes(data=True)
    if injected_uid in data["super_uids"]
)

print("raw test vertices:", test.number_of_nodes())
print("transformed test vertices:", H_test.number_of_nodes())
print("compressed vertex containing injected UID:", injected_token_node)


In [ ]:
edge_model, transformed_train = fit_transformed_markov_model(
    coarsener,
    train,
    alpha=8.0,
    base_pseudocount=0.5,
    weighting="edge",
)

parent_model = TreeMarkovModel(
    alpha=8.0,
    base_pseudocount=0.5,
    weighting="parent",
    label_attr=coarsener.label_attr,
    size_attr=coarsener.size_attr,
    time_attr=coarsener.time_attr,
).fit(transformed_train)

edge_scores = pd.DataFrame(
    record.to_record()
    for record in edge_model.score_vertices(
        H_test,
        rule_information=path.rule_by_token,
    )
)
parent_scores = pd.DataFrame(
    record.to_record()
    for record in parent_model.score_vertices(
        H_test,
        rule_information=path.rule_by_token,
    )
)

edge_scores["label_display"] = edge_scores["label"].map(format_token)
edge_scores["parent_label_display"] = edge_scores["parent_label"].map(
    lambda value: None if value is None else format_token(value)
)
edge_scores["contains_injected_uid"] = edge_scores["node"].eq(injected_token_node)
edge_scores["parent_balanced_transition_surprisal_bits"] = parent_scores.set_index("node").loc[
    edge_scores["node"], "transition_surprisal_bits"
].to_numpy()

columns = [
    "node",
    "label_display",
    "parent_label_display",
    "size",
    "contains_injected_uid",
    "transition_probability",
    "transition_surprisal_bits",
    "transition_surprisal_per_site_bits",
    "sibling_surprisal_bits",
    "rule_local_surprisal_bits",
    "rule_construction_surprisal_bits",
    "rule_construction_surprisal_per_site_bits",
    "parent_balanced_transition_surprisal_bits",
]
edge_scores.sort_values("transition_surprisal_bits", ascending=False)[columns].head(20).round(4)


Interpret the columns as separate diagnostics:

- **Transition surprisal:** how unexpected this occurrence is under its current
  transformed parent context.
- **Sibling surprisal:** the same child after allowing repeated siblings in this
  one parent to reinforce one another.
- **Rule-local / construction surprisal:** how common the token's learned merge
  recipe was on training data.  This is a token-type property, not a new-graph
  occurrence property.
- **Parent-balanced transition surprisal:** a robustness view in which one giant
  training hub cannot dominate hundreds of ordinary parents.

Do not automatically sum contextual and rule-construction surprisal: the rule table
is a shared model cost, whereas contextual surprise is paid per occurrence.


In [ ]:
annotated = edge_model.annotate_graph(
    H_test,
    rule_information=path.rule_by_token,
    prefix="info_",
    copy=True,
)

pos = nx.spring_layout(annotated, seed=7)
values = [
    annotated.nodes[node]["info_transition_surprisal_bits"]
    for node in annotated.nodes
]
labels = {
    node: format_token(annotated.nodes[node]["label"])
    for node in annotated.nodes
}

fig, ax = plt.subplots(figsize=(12, 8))
collection = nx.draw_networkx_nodes(
    annotated,
    pos,
    node_color=values,
    cmap="viridis",
    node_size=180,
    ax=ax,
)
nx.draw_networkx_edges(annotated, pos, arrows=True, alpha=0.45, ax=ax)
nx.draw_networkx_labels(annotated, pos, labels=labels, font_size=6, ax=ax)
ax.set_title("Transformed held-out tree: color = contextual surprisal (bits)")
ax.set_axis_off()
fig.colorbar(collection, ax=ax, label="bits")
plt.show()

edge_scores.loc[edge_scores["contains_injected_uid"], columns].round(4)


## 6. Computational cost

This experiment favors interpretability over speed.

- Fitting or scoring one `TreeMarkovModel` is approximately linear in the number
  of live vertices and edges.  Parent-balanced counts have the same asymptotic cost.
- Sibling-predictive scoring sorts each child group for deterministic per-child
  attribution, costing roughly $\sum_v d_v\log d_v$.  The total child-bag score is
  order invariant.
- Replaying $R$ fitted BPE rules is deliberately expensive: it performs $R$
  one-rule transformations and refits/rescores a probability model after each
  stage.  Use a held-out sample or `max_rules=` while exploring large corpora.
- None of this changes the fitted `tree_coarsening` module or its fast path.
